# EDA — Leads BFC

Análise exploratória do `leads_master.csv` para subsidiar o modelo de otimização de alocação de vendedores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # sem display interativo
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE, ORANGE, GREEN, RED = '#4878CF', '#E8912D', '#56A45E', '#D65F5F'

df = pd.read_csv('leads_master.csv', encoding='utf-8')
hist = df[df['particao'] == 'historico'].copy()
ativos = df[df['particao'] == 'ativo'].copy()

print(f'Total: {len(df)} | Histórico: {len(hist)} | Ativos: {len(ativos)}')

## 1. Completude por coluna

Coluna verde = ≥ 80% preenchida | Laranja = 50–79% | Vermelha = < 50%

In [ ]:
pct = (df.replace('', np.nan).notna().sum() / len(df) * 100).round(1).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = [RED if v < 50 else ORANGE if v < 80 else GREEN for v in pct]
bars = ax.barh(pct.index, pct.values, color=colors_bar)
ax.axvline(100, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Completude (%)')
ax.set_title('Figura 1 — Completude por coluna (n=113)')
for bar, val in zip(bars, pct.values):
    ax.text(min(val + 0.5, 96), bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('fig1_completude.png', dpi=150)
plt.show()

# Colunas críticas com baixa completude
print('\nColunas com completude < 60%:')
print(pct[pct < 60].to_string())

## 2. Variável alvo e viés de seleção

**Achado-chave:** O histórico tem 68 conversões e apenas 8 não-conversões (89% de taxa).  
Isso **não** significa que a equipe converte 89% dos leads — significa que leads perdidos raramente têm `interesse_nivel` preenchido,  
e portanto raramente entram no registro com desfecho. É um viés de seleção explicitado no dicionário de dados.

In [ ]:
hist['conv'] = pd.to_numeric(hist['converteu'], errors='coerce')
hist_def = hist[hist['conv'].notna()]
conv_counts = [int(hist_def['conv'].eq(1).sum()), int(hist_def['conv'].eq(0).sum())]
print(f'Com desfecho: {len(hist_def)} | Ganhos: {conv_counts[0]} | Perdidos: {conv_counts[1]}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].pie(conv_counts,
            labels=[f'Fechou Ganho (1)\nn={conv_counts[0]}',
                    f'Não Fechou (0)\nn={conv_counts[1]}'],
            colors=[GREEN, RED], autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title(f'Figura 2a — Variável alvo (histórico c/ desfecho, n={len(hist_def)})')

ordem = ['Muito Alto', 'Alto', 'Médio', 'Moderado', 'Baixo']
h_int = hist_def[hist_def['interesse_nivel'].notna() & (hist_def['interesse_nivel'] != '')]
taxa = (h_int.groupby('interesse_nivel')['conv'].mean() * 100).reindex(
    [i for i in ordem if i in h_int['interesse_nivel'].unique()])
n_nivel = h_int['interesse_nivel'].value_counts().reindex(taxa.index)

bars = axes[1].bar(taxa.index, taxa.values, color=BLUE, edgecolor='white')
axes[1].set_ylim(0, 120)
axes[1].set_ylabel('Taxa de conversão (%)')
axes[1].set_title('Figura 2b — Taxa por interesse_nivel\n(⚠ viés de seleção)')
axes[1].tick_params(axis='x', rotation=20)
for bar, n, val in zip(bars, n_nivel.values, taxa.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 2,
                 f'{val:.0f}%\n(n={n})', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig2_alvo_vies.png', dpi=150)
plt.show()

## 3. Variáveis do modelo: Pᶜ, Vt, w, eficiência

**Estatísticas descritivas (n=113):**

| | pc_estimado | vt_estimado_RS | w_peso_aresta | eficiencia_w_t |
|--|--|--|--|--|
| mean | 0.39 | R$ 3.863 | R$ 1.262 | 698 |
| median | 0.45 | R$ 3.000 | R$ 1.350 | 675 |
| max | 0.85 | R$ 12.000 | R$ 7.800 | 2.600 |

A distribuição de `pc_estimado` é bimodal: pico em 0.10 (leads sem interesse declarado) e pico em 0.45–0.65 (leads com interesse).

In [ ]:
variaveis = [
    ('pc_estimado',    'Pᶜ — Probabilidade de conversão',   BLUE),
    ('vt_estimado_RS', 'Vt — Valor do contrato (R$)',        ORANGE),
    ('w_peso_aresta',  'w = Pᶜ × Vt — Valor esperado (R$)', GREEN),
    ('eficiencia_w_t', 'w/t — Eficiência (R$/hora)',         RED),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, titulo, cor) in zip(axes.flat, variaveis):
    dados = df[col].dropna()
    ax.hist(dados, bins=20, color=cor, edgecolor='white', alpha=0.85)
    ax.axvline(dados.mean(),   color='black', lw=1.5, ls='--', label=f'Média={dados.mean():.1f}')
    ax.axvline(dados.median(), color='grey',  lw=1.2, ls=':',  label=f'Mediana={dados.median():.1f}')
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequência')
plt.suptitle('Figura 3 — Distribuição das variáveis do modelo (n=113)', fontsize=12)
plt.tight_layout()
plt.savefig('fig3_variaveis_modelo.png', dpi=150)
plt.show()

print(df[['pc_estimado','vt_estimado_RS','w_peso_aresta','eficiencia_w_t']].describe().round(2))

## 4. Análise por vendedor

**Achados:**
- Marcela Belo concentra 46/84 leads com responsável (55%) e 100% de conversão no histórico
- Talita Guedes tem 80% — único vendedor com conversão < 100% no histórico
- Taxa de 100% nos demais deve ser lida com cautela: n pequeno (Karoline n=3, Eduarda n=1)
- Fator de afinidade A(li, vj) não está modelado ainda — todos leads assumem A=1

In [ ]:
vend = df[df['responsavel'].notna() & (df['responsavel'] != '')].copy()
vend_hist = vend[vend['particao'] == 'historico'].copy()
vend_hist['conv_num'] = pd.to_numeric(vend_hist['converteu'], errors='coerce')

resumo = vend_hist.groupby('responsavel').agg(
    n_hist=('lead_id', 'count'),
    taxa_conv=('conv_num', 'mean'),
    w_total=('w_peso_aresta', 'sum'),
).sort_values('w_total', ascending=False)
resumo['taxa_conv_pct'] = (resumo['taxa_conv'] * 100).round(1)
print('Resumo por vendedor (histórico):')
print(resumo[['n_hist','taxa_conv_pct','w_total']])

n_total_vend = vend.groupby('responsavel').size().sort_values(ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].barh(n_total_vend.index, n_total_vend.values, color=BLUE)
axes[0].set_title('Figura 4a — Leads por vendedor (total)')
axes[0].set_xlabel('Nº de leads')
for i, v in enumerate(n_total_vend.values):
    axes[0].text(v + 0.2, i, str(v), va='center', fontsize=9)

tc = resumo['taxa_conv_pct'].sort_values()
axes[1].barh(tc.index, tc.values, color=GREEN)
axes[1].set_title('Figura 4b — Taxa de conversão (%)\nhistórico')
axes[1].set_xlabel('%')
for i, v in enumerate(tc.values):
    axes[1].text(v + 0.3, i, f'{v}%', va='center', fontsize=9)

wt = resumo['w_total'].sort_values()
axes[2].barh(wt.index, wt.values, color=ORANGE)
axes[2].set_title('Figura 4c — Valor esperado total (w)\npor vendedor — histórico')
axes[2].set_xlabel('R$ esperado')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
for i, v in enumerate(wt.values):
    axes[2].text(v + 50, i, f'R${v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig4_vendedores.png', dpi=150)
plt.show()

## 5. Subconjunto ativo — 37 leads para otimização

**Achados críticos para a otimização:**
- `pc_estimado` mediano dos ativos = 0.07 (muito abaixo dos históricos = 0.45) — a maioria são leads sem interesse declarado
- Lead 'Simone' (PJ, Alto) é claramente o prioritário: w=R$7.800, eficiência=2.600 R$/h — mas está **sem vendedor atribuído**
- 8 leads sem responsável no subconjunto ativo = oportunidade direta de ganho com a otimização
- 100% dos ativos têm Vt estimado por segmento (nenhum valor real)

In [ ]:
ativos['tem_resp'] = ativos['responsavel'].apply(
    lambda x: 'Com vendedor' if (pd.notna(x) and x != '') else 'Sem vendedor')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(ativos['eficiencia_w_t'], bins=15, color=BLUE, edgecolor='white')
axes[0].axvline(ativos['eficiencia_w_t'].mean(), color=RED, lw=1.5, ls='--',
                label=f"Média={ativos['eficiencia_w_t'].mean():.0f}")
axes[0].set_title('Figura 5a — Eficiência (w/t)\nleads ativos')
axes[0].set_xlabel('R$ esperado / hora')
axes[0].legend(fontsize=9)

qd = ativos['qualidade_dado'].value_counts()
col_map = {'Media': ORANGE, 'Baixa': RED, 'Alta': GREEN}
axes[1].pie(qd, labels=qd.index, autopct='%1.1f%%',
            colors=[col_map.get(i, BLUE) for i in qd.index], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Figura 5b — Qualidade do dado\nleads ativos')

rc = ativos['tem_resp'].value_counts()
bar_colors = [GREEN if idx == 'Com vendedor' else RED for idx in rc.index]
axes[2].bar(rc.index, rc.values, color=bar_colors, edgecolor='white')
axes[2].set_title('Figura 5c — Atribuição de vendedor\nleads ativos')
axes[2].set_ylabel('Nº leads')
for i, v in enumerate(rc.values):
    axes[2].text(i, v + 0.2, str(v), ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Figura 5 — Leads ativos (n=37) — alvo da otimização', fontsize=12)
plt.tight_layout()
plt.savefig('fig5_ativos.png', dpi=150)
plt.show()

print('\nTop 10 ativos por eficiência (candidatos prioritários):')
cols = ['nome','tipo_cliente','interesse_nivel','responsavel',
        'w_peso_aresta','eficiencia_w_t','qualidade_dado']
print(ativos.nlargest(10,'eficiencia_w_t')[cols].to_string(index=False))

## 6. Sumário executivo da EDA

| Dimensão | Achado |
|---|---|
| Dataset | 113 leads: 76 histórico + 37 ativos |
| Variável alvo | 89% de conversão no histórico — **viés de seleção**, não performance real |
| Pc estimado | 41% dos leads usam `prior_sem_dado` (pc=0.10); restante usa prior por interesse |
| Vt | 99,1% estimado por segmento (PF=R$2.500–3.000, PJ=R$12.000); apenas 1 lead com valor real |
| Vendedores | Marcela Belo concentra 55% dos leads com responsável; 8 ativos sem vendedor |
| Lead prioritário | 'Simone' (PJ, Alto interesse): w=R$7.800, efic=2.600 R$/h — **sem vendedor atribuído** |
| Qualidade | 100% dos ativos têm qualidade Baixa ou Média — nenhum tem Vt real |

**Implicações para o artigo:**
- Os resultados da otimização são sensíveis às estimativas de Pᶜ e Vt; análise de sensibilidade é recomendada
- O modelo Knapsack pode ser rodado nas simulações mas deve ser apresentado como exercício metodológico
- O ganho mais imediato e concreto da otimização é a **alocação dos 8 leads ativos sem vendedor**